# Backend Endpoint Functionality Tests

This notebook tests your local FastAPI backend using `requests`.

- `GET /health`
- `POST /chat/stream` (NDJSON stream)

In [28]:
import json
import requests

BASE_URL = "http://127.0.0.1:8000"
TIMEOUT_SECONDS = 120

print("Using backend:", BASE_URL)

Using backend: http://127.0.0.1:8000


In [29]:
# Health endpoint test
health_resp = requests.get(f"{BASE_URL}/health", timeout=30)
print("Status:", health_resp.status_code)
print("Body:", health_resp.json())

Status: 200
Body: {'status': 'ok', 'app': 'target-dashboard-agent'}


In [30]:
mock_frontend_payload = {
    "messages": [
        {
            "role": "user",
            "content": "Please inspect the database first, then answer follow-up questions briefly.",
        },
        {
            "role": "assistant",
            "content": "I will inspect the available tables and schema before answering.",
        },
        {
            "role": "user",
            "content": "How many distinct Commitment Types are there?",
        },
    ]
}

mock_frontend_payload

{'messages': [{'role': 'user',
   'content': 'Please inspect the database first, then answer follow-up questions briefly.'},
  {'role': 'assistant',
   'content': 'I will inspect the available tables and schema before answering.'},
  {'role': 'user',
   'content': 'How many distinct Commitment Types are there?'}]}

In [31]:
def _short_text(value, max_len=220):
    text = str(value).strip()
    if len(text) <= max_len:
        return text
    return text[: max_len - 3] + "..."


def print_agent_stream_for_user(response):
    """Render backend NDJSON as user-friendly frontend output."""
    for raw_line in response.iter_lines(decode_unicode=True):
        if not raw_line:
            continue

        event = json.loads(raw_line)
        event_type = event.get("event")
        data = event.get("data", {})

        if event_type == "tool_call":
            print(f"Agent is using `{data.get('tool')}` with {data.get('args')}")
        elif event_type == "tool_result":
            print(f"Tool `{data.get('tool')}` returned: {_short_text(data.get('content'))}")
        elif event_type == "final":
            print(f"\nAnswer: {data.get('text')}")
        elif event_type == "error":
            print(f"\nError: {data.get('detail') or data.get('message')}")
        else:
            print(f"Unhandled event: {event}")

In [34]:
with requests.post(
    f"{BASE_URL}/chat/stream",
    json=mock_frontend_payload,
    timeout=TIMEOUT_SECONDS,
    stream=True,
) as response:
    print("Status:", response.status_code)
    response.raise_for_status()
    print_agent_stream_for_user(response)

Status: 200
Agent is using `sql_db_list_tables` with {}
Tool `sql_db_list_tables` returned: Worksheet
Agent is using `sql_db_schema` with {'table_names': 'Worksheet'}
Tool `sql_db_schema` returned: CREATE TABLE "Worksheet" (
	"Company_Name" TEXT, 
	"ISIN" TEXT, 
	"LEI" TEXT, 
	"Location" TEXT, 
	"Region" TEXT, 
	"Sector" TEXT, 
	"Organization_Type" TEXT, 
	"Action" TEXT, 
	"Commitment_Type" TEXT, 
	"Commitment_D...
Agent is using `sql_db_query_checker` with {'query': 'SELECT COUNT(DISTINCT Commitment_Type) FROM Worksheet'}
Tool `sql_db_query_checker` returned: SELECT COUNT(DISTINCT Commitment_Type) FROM Worksheet
Agent is using `sql_db_query` with {'query': 'SELECT COUNT(DISTINCT Commitment_Type) FROM Worksheet'}
Tool `sql_db_query` returned: [(4,)]

Answer: There are 4 distinct Commitment Types.
